#  About the Datasets and Problem Statement

Imagine you’re designing a system that helps users discover the right products from thousands of options — just like how Amazon or Netflix recommends what you’ll love next!  
In this assignment, you’ll explore multiple approaches to building **Recommender Systems**, using real-world datasets that capture customer behavior, product details, and review patterns.



##  Dataset 1: Online Retail Dataset (Tasks 1–3)

This dataset captures **real transactions** from a UK-based online retail store and forms the foundation for the first three tasks.  
Each record represents a product purchase, with details about the product, customer, and transaction context.

### Key Attributes:
- **Invoice**: Unique ID for each transaction.  
- **StockCode**: Identifier for each product.  
- **Description**: Name or description of the product.  
- **Quantity**: Number of items purchased per transaction.  
- **InvoiceDate**: Date and time of the transaction.  
- **Price**: Price per item in pounds (£).  
- **Customer ID**: Unique ID for each customer.  
- **Country**: Customer’s country of residence.



##  Dataset 2: Amazon Fine Food Reviews (Task 4)

This dataset contains **customer reviews** of food products sold on Amazon, including textual feedback and numerical ratings.  

### Key Attributes:
- **UserId**: Unique identifier for each reviewer.  
- **ProductId**: Unique identifier for each product.  
- **Score**: Rating given by the user (typically on a scale of 1–5).  
(Ignore Other Columns)

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Embedding, Flatten, Dot, Dense, Concatenate
from tensorflow.keras.models import Model
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer, util
from numpy.linalg import norm

2026-09-30 01:30:32.211591: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790731832.504612      40 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790731832.595971      40 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [2]:
# Load the online-retail-dataset
df1 = pd.read_excel('/kaggle/input/online-retail-dataset/online_retail_II.xlsx')
df1.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Task 1: EDA and Preprocessing ( 1 mark )

In this task, you are required to perform **data exploration** and **preprocessing** to understand the characteristics of the dataset and prepare it for further modeling in subsequent tasks.


- Understand the **structure and quality** of the dataset.  
- Identify and **handle missing** values.  
- Perform **basic preprocessing** for both numerical and categorical attributes.  


### **Things we want you to do:**

- Convert `InvoiceDate` into a proper **datetime** format.    
- Convert categorical attributes (e.g., `Country`, `StockCode`) to a suitable data type (`category` or `object`).  
- Normalize numeric columns like **Quantity** and **Price** if needed (using **Standard-scaling**).
- Pre-process the **text fields** ( strip any punctuations, convert them to lower case etc)

In [3]:
import re
from sklearn.preprocessing import StandardScaler

# Display basic information
print("Dataset Info:")
print(df1.info())
print("\nMissing Values:")
print(df1.isnull().sum())
print("\nBasic Statistics:")
print(df1.describe())

# Handle missing values
print(f"\nRows before cleaning: {len(df1)}")
df1 = df1.dropna(subset=['Customer ID'])  # Remove rows without customer ID
df1 = df1.dropna(subset=['Description'])  # Remove rows without description
print(f"Rows after cleaning: {len(df1)}")

# Convert InvoiceDate to datetime
df1['InvoiceDate'] = pd.to_datetime(df1['InvoiceDate'])

# Convert categorical columns
df1['Country'] = df1['Country'].astype('category')
df1['StockCode'] = df1['StockCode'].astype('str')
df1['Customer ID'] = df1['Customer ID'].astype('str')

# Preprocess text fields - clean Description
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', ' ', text)  # Remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()  # Remove extra spaces
    return text

df1['Description_Clean'] = df1['Description'].apply(clean_text)

# Normalize numerical columns (Quantity and Price)
scaler = StandardScaler()
df1['Quantity_Normalized'] = scaler.fit_transform(df1[['Quantity']])
df1['Price_Normalized'] = scaler.fit_transform(df1[['Price']])

print("\nPreprocessed Dataset:")
print(df1.head())
print("\nData types after preprocessing:")
print(df1.dtypes)

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[ns]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 32.1+ MB
None

Missing Values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

Basic Statistics:
            Quantity                    InvoiceDate          Pri

/usr/local/lib/python3.11/dist-packages/pandas/core/algorithms.py:1743: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  return lib.map_infer(values, mapper, convert=convert)
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:605: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype):
/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:614: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  if is_sparse(pd_dtype) or not is_extension_array_dtype(pd_dtype):
/usr/local/lib/python3.11/dist-packages/pandas/core/algorithms.py:1743: DeprecationWarning: is_sparse is deprecated and will be removed in a future version. Check `isinstance(dtype, pd.SparseDtype)` instead.
  return lib.map

## Task 2: Knowledge-Based Recommender System ( 2 marks )

In this task, you are required to build a **case-based recommender system** for the **Online Retail dataset**.

- Develop a system that takes **user preferences** as input:  
  - Preferred **Country**  
  - Product **Category/Description keyword**  
  - **Maximum Budget**  
- Use a **similarity-based retrieval approach**  to find relevant products.  
- Recommend the **Top 5 matching products** to the user.



### **Things we want you to do**

- **Preprocess** the dataset by selecting essential columns like **StockCode**, **Description**, **Country**, and **Price**, and handle missing values.  
- Use **any form of embedding** - Sentence Transformers, BERT,GLOVE, Word2Vec etc to **embed product descriptions** and compute **cosine similarity** for text-based matching.  
- Apply **Manhattan** or **Euclidean distance** for comparing **numerical features** such as price or budget.  
- **Normalize** each distance type (**text**, **numeric**, **categorical**) individually to the range **[0, 1]** before combining.  
- **Aggregate** the normalized distances using suitable **weights** and recommend the **Top 5 products** with the smallest overall distance.

<br>

**Make recommendations for a user with the following preferences:**

- **Country:** United Kingdom  
- **Product Keyword:** Lights  
- **Price:** Any price  


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import euclidean, cityblock

# Prepare dataset for recommendations
df_rec = df1[['StockCode', 'Description', 'Description_Clean', 'Country', 'Price']].copy()
df_rec = df_rec.dropna()
df_rec = df_rec.drop_duplicates(subset=['StockCode'])
df_rec = df_rec.reset_index(drop=True)

print(f"Products available for recommendation: {len(df_rec)}")

# Create TF-IDF vectorizer for text embeddings
print("Creating TF-IDF embeddings for product descriptions...")
tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
product_embeddings = tfidf.fit_transform(df_rec['Description_Clean'])

print(f"TF-IDF matrix shape: {product_embeddings.shape}")

def recommend_products(user_country, product_keyword, max_price=None, top_n=5):
    """
    Knowledge-based recommender using hybrid similarity
    """
    # 1. Text Similarity (using TF-IDF embeddings)
    keyword_clean = clean_text(product_keyword)
    keyword_embedding = tfidf.transform([keyword_clean])
    text_similarity = cosine_similarity(keyword_embedding, product_embeddings)[0]
    # Convert to distance (1 - similarity)
    text_distance = 1 - text_similarity
    
    # 2. Categorical Similarity
    def country_distance(country1, country2):
        if str(country1).strip().lower() == str(country2).strip().lower():
            return 0.0
        else:
            return 1.0
    
    country_distances = df_rec['Country'].apply(
        lambda x: country_distance(user_country, x)
    ).values
    
    # 3. Numerical Similarity (Price)
    if max_price is not None:
        max_price_val = max(df_rec['Price'].max(), max_price)
        price_distances = df_rec['Price'].apply(
            lambda x: abs(x - max_price) / max_price_val if max_price_val > 0 else 0
        ).values
    else:
        price_distances = np.zeros(len(df_rec))
    
    # Normalize all distances to [0, 1]
    text_distance_norm = (text_distance - text_distance.min()) / (text_distance.max() - text_distance.min() + 1e-10)
    country_distance_norm = country_distances  # Already 0 or 1
    price_distance_norm = price_distances  # Already normalized
    
    # Combine distances with weights
    weight_text = 0.5
    weight_country = 0.3
    weight_price = 0.2
    
    total_distance = (weight_text * text_distance_norm + 
                     weight_country * country_distance_norm + 
                     weight_price * price_distance_norm)
    
    # Get top N recommendations
    top_indices = np.argsort(total_distance)[:top_n]
    
    recommendations = df_rec.iloc[top_indices][['StockCode', 'Description', 'Country', 'Price']].copy()
    recommendations['Similarity_Score'] = 1 - total_distance[top_indices]
    
    return recommendations

# Make recommendations
print("\n" + "="*60)
print("RECOMMENDATIONS FOR USER")
print("="*60)
print("Country: United Kingdom")
print("Product Keyword: Lights")
print("Price: Any price")
print("="*60)

recommendations = recommend_products(
    user_country='United Kingdom',
    product_keyword='Lights',
    max_price=None,
    top_n=5
)

print("\nTop 5 Recommended Products:")
print(recommendations.to_string(index=False))

Products available for recommendation: 4031
Creating TF-IDF embeddings for product descriptions...
TF-IDF matrix shape: (4031, 1000)

RECOMMENDATIONS FOR USER
Country: United Kingdom
Product Keyword: Lights
Price: Any price

Top 5 Recommended Products:
StockCode                         Description        Country  Price  Similarity_Score
    79321                       CHILLI LIGHTS United Kingdom   4.95          1.000000
    72586 S/6 HALLOWEEN GHOST T LIGHTS, 5CM H United Kingdom   0.42          1.000000
    79341               WILLOW BRANCH LIGHTS. United Kingdom   6.75          1.000000
    79320                     FLAMINGO LIGHTS United Kingdom   4.95          1.000000
   79340W          WHITE ORCHID FLOWER LIGHTS United Kingdom   7.95          0.837219


## Task 3: Market Basket Analysis ( 2 marks )

In this task, you are required to perform **association rule mining** on the transaction data using the **Apriori algorithm**.

- Convert the transaction dataset into a **one-hot encoded format** suitable for association rule mining.
- Find **frequent itemsets** using a **support threshold of 0.02**.
- Generate **association rules** with **confidence ≥ 0.3**.
- Interpret and explain the **top 5 rules** in terms of business insights.

###  **Things we want you to do**

- Preprocess the dataset to group items by **InvoiceNo**.
- Ensure the one-hot encoding reflects **presence/absence** of each product in a transaction.
- Consider metrics like **support, confidence, and lift** to evaluate rules.
- Comment your findings and possible insights from the top rules.

NOTE: You can make use of the apriori, association_rules function from the mlxtend library in python

In [5]:
# Prepare transaction data
transactions = df1.groupby('Invoice')['Description_Clean'].apply(list).values.tolist()

# Remove empty descriptions
transactions = [[item for item in transaction if item != ""] for transaction in transactions]
transactions = [transaction for transaction in transactions if len(transaction) > 0]

print(f"\nTotal transactions: {len(transactions)}")
print(f"Sample transaction: {transactions[0][:5]}")

# One-hot encode transactions
te = TransactionEncoder()
te_ary = te.fit(transactions).transform(transactions)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

print(f"\nOne-hot encoded shape: {df_encoded.shape}")

# Apply Apriori algorithm
print("\nFinding frequent itemsets...")
frequent_itemsets = apriori(df_encoded, min_support=0.02, use_colnames=True)
print(f"Frequent itemsets found: {len(frequent_itemsets)}")

# Generate association rules
print("\nGenerating association rules...")
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.3)
rules = rules.sort_values('lift', ascending=False)

print(f"Association rules found: {len(rules)}")

# Display top 5 rules
print("\n" + "="*80)
print("TOP 5 ASSOCIATION RULES")
print("="*80)
top_5_rules = rules.head(5)
for idx, rule in top_5_rules.iterrows():
    print(f"\nRule {idx + 1}:")
    print(f"  Antecedent: {list(rule['antecedents'])}")
    print(f"  Consequent: {list(rule['consequents'])}")
    print(f"  Support: {rule['support']:.4f}")
    print(f"  Confidence: {rule['confidence']:.4f}")
    print(f"  Lift: {rule['lift']:.4f}")
    print(f"  Interpretation: If a customer buys {list(rule['antecedents'])}, ")
    print(f"                  they are {rule['confidence']*100:.1f}% likely to also buy {list(rule['consequents'])}")





Total transactions: 23587
Sample transaction: ['15cm christmas glass ball 20 lights', 'pink cherry lights', 'white cherry lights', 'record frame 7 single size', 'strawberry ceramic trinket box']

One-hot encoded shape: (23587, 4362)

Finding frequent itemsets...
Frequent itemsets found: 126

Generating association rules...
Association rules found: 11

TOP 5 ASSOCIATION RULES

Rule 11:
  Antecedent: ['wooden picture frame white finish']
  Consequent: ['wooden frame antique white']
  Support: 0.0237
  Confidence: 0.6723
  Lift: 15.3954
  Interpretation: If a customer buys ['wooden picture frame white finish'], 
                  they are 67.2% likely to also buy ['wooden frame antique white']

Rule 10:
  Antecedent: ['wooden frame antique white']
  Consequent: ['wooden picture frame white finish']
  Support: 0.0237
  Confidence: 0.5417
  Lift: 15.3954
  Interpretation: If a customer buys ['wooden frame antique white'], 
                  they are 54.2% likely to also buy ['wooden pictur

BUSINESS INSIGHTS:
1. Products with high lift (>1) indicate strong positive correlation
2. High confidence rules can be used for product bundling strategies
3. These associations help in store layout and cross-selling
4. Support indicates how frequently items appear together
5. Rules can inform inventory management and promotional strategies

[](http://)